# Notary 662 - RAG Test Panel (Llama 3)

This notebook allows you to test the Retrieval-Augmented Generation (RAG) system connected to Notary Office 662's dataset.

## 🚀 Getting Started

1.  **HF_TOKEN**: You need a Hugging Face token with access to the `meta-llama/Llama-3.2-3B-Instruct` model.
2.  **GOOGLE_API_KEY**: Required if using Gemini features.

Add these to your Colab Secrets (Left sidebar > 🔑 icon).

In [ ]:
!pip install langchain langchain-community langchain-huggingface faiss-cpu pypdf requests

In [ ]:
import os
import requests
from google.colab import userdata

# 1. Setup API Keys
# It is recommended to add 'HF_TOKEN' to your Colab Secrets
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except:
    HF_TOKEN = "YOUR_HF_TOKEN_HERE"

CLOUDFLARE_URL = "https://notary-662-sbz.pages.dev/api/db/chats"

from langchain_huggingface import HuggingFaceEndpoint
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA

# 2. Load Llama 3 Model
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.2-3B-Instruct",
    huggingfacehub_api_token=HF_TOKEN,
    temperature=0.5,
    max_new_tokens=512
)

# 3. Setup Persian Embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

print("Model and Embeddings Loaded Successfully.")

### Run a Test Query

This will query the model and automatically sync the result to your Cloudflare D1 history.

In [ ]:
import os

def ask_notary_ai(question):
    prompt = f"Context: [Notary 662 Legal Archive]\nUser Question: {question}\nAssistant (Official Persian):"
    response = llm.invoke(prompt)
    
    # Sync with Cloudflare
    chat_data = {
        "id": "colab-test-" + os.urandom(3).hex(),
        "title": "Query from Colab",
        "docType": "legal_research",
        "messages": [
            {"role": "user", "text": question},
            {"role": "model", "text": response}
        ]
    }
    try:
        res = requests.post(CLOUDFLARE_URL, json=chat_data)
        print(f"Synced to Cloudflare: {res.status_code}")
    except:
        print("Cloudflare Sync Failed.")
        
    return response

query = "شرایط تنظیم صلح نامه رسمی در دفترخانه ۶۶۲ چیست؟"
print(ask_notary_ai(query))